## **Installs**

In [ ]:
#!pip install -U transformers

## **Hugging Face Token**

In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("my_hf_token")

In [2]:
from huggingface_hub import login
login(token=hf_token)

## **Imports**

In [3]:
import json
import re
import math
import random
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers
import gc, traceback

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# **Part 1**

## **Negative Responses Generation**

In [4]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
HF_TOKEN = hf_token

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)   # no token needed, it's public
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto",
)
model.eval()

DEVICE = next(model.parameters()).device
NUM_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size
print(f"Loaded {MODEL_NAME}: {NUM_LAYERS} layers, hidden_dim={HIDDEN_DIM}, device={DEVICE}")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-7B-Instruct: 28 layers, hidden_dim=3584, device=cuda:0


In [6]:
#freeze model params

for p in model.parameters():
    p.requires_grad_(False)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params in base model (should be 0): {trainable}")

Trainable params in base model (should be 0): 0


In [7]:
df = pd.read_csv("/kaggle/input/datasets/anarvaaa/dataset-for-ssv/combined_pico_questions.csv")  # update path if running locally

print(df.shape)
df.head(3)

(100, 7)


,sr.no.,speciality,question,P,I,C,O
0,1,Cardiology,In adults with heart failure with preserved ej...,Adults with HFpEF (LVEF ≥ 50%),"SGLT2 inhibitors (e.g., empagliflozin, dapagli...",Placebo + standard care,Composite of cardiovascular death and hospital...
1,2,Cardiology,In patients undergoing percutaneous coronary i...,Adults undergoing PCI for acute coronary syndrome,Short-term DAPT (1–3 mo) followed by P2Y12 inh...,Standard DAPT for 12 months,Bleeding Academic Research Consortium (BARC) t...
2,3,Cardiology,In adults with severe asymptomatic aortic sten...,Adults with severe asymptomatic aortic stenosis,Early elective AVR (surgical or transcatheter),Conservative watchful waiting,All-cause mortality; myocardial infarction at ...


In [8]:
GEN_MAX_NEW_TOKENS = 100
GEN_DO_SAMPLE = False
GEN_TEMPERATURE = 1.0
GEN_TOP_P = 0.95

In [9]:
# code to generate negative responses from the model
@torch.no_grad()
def generate_negative(question, max_new_tokens=GEN_MAX_NEW_TOKENS):
    messages = [{"role": "user", "content": question.strip()}]
    chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat_text, return_tensors="pt").to(DEVICE)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=GEN_DO_SAMPLE,
        temperature=GEN_TEMPERATURE,
        top_p=GEN_TOP_P,
        pad_token_id=tokenizer.pad_token_id,
    )
    text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return text.strip()

negative_responses = []
for i, row in df.iterrows():
    neg = generate_negative(row["question"])
    negative_responses.append(neg)
    if (i + 1) % 5 == 0 or (i + 1) == len(df):
        print(f"[{i+1}/{len(df)}] done")
        print("  Q:", row["question"][:100])
        print("  neg:", neg[:150].replace("\n", " "))

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[5/100] done
  Q: [Diagnostic] What is the diagnostic test accuracy of coronary computed tomography angiography (CCTA)
  neg: Coronary Computed Tomography Angiography (CCTA) is a non-invasive imaging technique used to visualize the coronary arteries and detect the presence an
[10/100] done
  Q: [Prognostic] Is high tumor mutational burden (TMB ≥ 10 mutations/megabase), compared with low TMB ($
  neg: High tumor mutational burden (TMB) is a biomarker that has been studied extensively in the context of immunotherapy, particularly with immune checkpoi
[15/100] done
  Q: [Diagnostic] What is the diagnostic test accuracy of cerebrospinal fluid (CSF) Aβ42​/Aβ40​ ratio com
  neg: The diagnostic test accuracy of cerebrospinal fluid (CSF) Aβ42/Aβ40 ratio compared to amyloid positron emission tomography (PET) imaging for diagnosin
[20/100] done
  Q: [Prognostic] Is an elevated serum procalcitonin level ($ \ge 0.5ng/mL)atpresentation,comparedwithlow
  neg: Elevated serum procalcitonin (PCT) level

In [11]:
import re

# Regex matching Chinese character ranges
chinese_pattern = re.compile(r'[\u4e00-\u9fff]')

# Fallback clean response map for identified queries
fallbacks = {
    "vonoprazan": (
        "Vonoprazan is a potassium-competitive acid blocker (P-CAB) providing "
        "potent acid suppression. Regimens combining vonoprazan and amoxicillin "
        "have shown non-inferior or superior H. pylori eradication rates compared "
        "to standard PPI-based triple therapy."
    )
}

cleaned_responses = []
for i, (q, resp) in enumerate(zip(df["question"], negative_responses)):
    if chinese_pattern.search(resp):
        print(f"Row {i+1} contained Chinese characters. Replacing...")
        # Check for matching keyword fallback or assign a generic clean narrative
        matched = False
        for kw, fallback_text in fallbacks.items():
            if kw.lower() in q.lower():
                cleaned_responses.append(fallback_text)
                matched = True
                break
        if not matched:
            cleaned_responses.append(
                "This intervention has been evaluated in clinical studies comparing "
                "its efficacy against standard care control groups across primary outcomes."
            )
    else:
        cleaned_responses.append(resp)

df_out = df.copy()
df_out["negative_response"] = cleaned_responses

Row 63 contained Chinese characters. Replacing...
Row 65 contained Chinese characters. Replacing...


In [12]:
# code which adds the positive repsonces and negative responses and create a new dataset -> combined_pico_pos_neg.csv
# save this in kaggle/working
# before moving to part 2, this will be uploaded into the dataset entity so it can be used later

out_path = "/kaggle/working/combined_pico_pos_neg.csv"
df_out.to_csv(out_path, index=False)
print(f"Saved {df_out.shape} to {out_path}")
df_out.head(3)

Saved (100, 8) to /kaggle/working/combined_pico_pos_neg.csv


,sr.no.,speciality,question,P,I,C,O,negative_response
0,1,Cardiology,In adults with heart failure with preserved ej...,Adults with HFpEF (LVEF ≥ 50%),"SGLT2 inhibitors (e.g., empagliflozin, dapagli...",Placebo + standard care,Composite of cardiovascular death and hospital...,The use of SGLT2 inhibitors in adults with hea...
1,2,Cardiology,In patients undergoing percutaneous coronary i...,Adults undergoing PCI for acute coronary syndrome,Short-term DAPT (1–3 mo) followed by P2Y12 inh...,Standard DAPT for 12 months,Bleeding Academic Research Consortium (BARC) t...,The question pertains to the optimal duration ...
2,3,Cardiology,In adults with severe asymptomatic aortic sten...,Adults with severe asymptomatic aortic stenosis,Early elective AVR (surgical or transcatheter),Conservative watchful waiting,All-cause mortality; myocardial infarction at ...,The question pertains to the outcomes of early...


# **Part 2: Experiments**

## **Load Model**

In [4]:
MODEL_NAME = "google/gemma-2b-it"
HF_TOKEN = hf_token

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    dtype=torch.float16,      # NOT torch_dtype (deprecated, as you already found)
    device_map="auto",
)
model.eval()

DEVICE = next(model.parameters()).device
NUM_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size
print(f"Loaded {MODEL_NAME}: {NUM_LAYERS} layers, hidden_dim={HIDDEN_DIM}, device={DEVICE}")

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Loaded google/gemma-2b-it: 18 layers, hidden_dim=2048, device=cuda:0


In [6]:
#freeze model params

for p in model.parameters():
    p.requires_grad_(False)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params in base model (should be 0): {trainable}")

Trainable params in base model (should be 0): 0


## **Load Data**

In [31]:
df = pd.read_csv("/kaggle/input/datasets/anarvaaa/dataset-for-ssv/combined_pico_pos_neg.csv")  # update path if running locally
print(df.shape)
df.head(3)

(100, 8)


,sr.no.,speciality,question,P,I,C,O,negative_response
0,1,Cardiology,In adults with heart failure with preserved ej...,Adults with HFpEF (LVEF ≥ 50%),"SGLT2 inhibitors (e.g., empagliflozin, dapagli...",Placebo + standard care,Composite of cardiovascular death and hospital...,The use of SGLT2 inhibitors in adults with hea...
1,2,Cardiology,In patients undergoing percutaneous coronary i...,Adults undergoing PCI for acute coronary syndrome,Short-term DAPT (1–3 mo) followed by P2Y12 inh...,Standard DAPT for 12 months,Bleeding Academic Research Consortium (BARC) t...,The question pertains to the optimal duration ...
2,3,Cardiology,In adults with severe asymptomatic aortic sten...,Adults with severe asymptomatic aortic stenosis,Early elective AVR (surgical or transcatheter),Conservative watchful waiting,All-cause mortality; myocardial infarction at ...,The question pertains to the outcomes of early...


In [32]:
PICO_INSTRUCTION = """You are given a clinical question.

Extract the question into the PICO framework:

P = Population/Patient: the patient population or condition being studied.
I = Intervention/Exposure: the treatment, intervention, or exposure being studied.
C = Comparator: the comparison treatment, control, or alternative.
O = Outcome: the clinical outcome or result of interest.

Return exactly four fields:
P: ...
I: ...
C: ...
O: ...
"""

In [33]:
def format_positive(row):
    return (
        f"P: {row['P']}\n"
        f"I: {row['I']}\n"
        f"C: {row['C']}\n"
        f"O: {row['O']}"
    )

# Whether to wrap the raw question in Gemma's chat turn markers. Deliberately does
# NOT add any "extract this into PICO format" instruction -- that would let the
# prompt itself do the steering, confounding whether the *vector* is responsible
# for the PICO-shaped output. The only thing IT models get beyond the base-model
# prompt is the turn structure they were tuned to expect; the content is identical.
IS_INSTRUCT_MODEL = "-it" in MODEL_NAME.lower()

'''
def format_prompt(row):
    question = row["question"].strip()

    return (
        "<start_of_turn>user\n"
        f"{PICO_INSTRUCTION}\n\n"
        f"Clinical question:\n{question}"
        "<end_of_turn>\n"
        "<start_of_turn>model\n"
    )
'''

def format_prompt(row):
    question = row["question"].strip()

    messages = [
        {
            "role": "user",
            "content": (
                f"{PICO_INSTRUCTION}\n\n"
                f"Clinical question:\n{question}"
            )
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

examples = []
for _, row in df.iterrows():
    examples.append({
        "prompt": format_prompt(row),
        "positive": format_positive(row) + tokenizer.eos_token,
        "negative": str(row["negative_response"]).strip() + tokenizer.eos_token,
    })

random.shuffle(examples)
n_val = max(5, int(0.1 * len(examples)))
val_examples = examples[:n_val]
train_examples = examples[n_val:]
print(f"train={len(train_examples)}  val={len(val_examples)}")
print(f"IS_INSTRUCT_MODEL={IS_INSTRUCT_MODEL}")
print("--- sample ---")
print("PROMPT:", train_examples[0]["prompt"])
print("POSITIVE:", train_examples[0]["positive"])
print("NEGATIVE:", train_examples[0]["negative"])


train=90  val=10
IS_INSTRUCT_MODEL=True
--- sample ---
PROMPT: <bos><start_of_turn>user
You are given a clinical question.

Extract the question into the PICO framework:

P = Population/Patient: the patient population or condition being studied.
I = Intervention/Exposure: the treatment, intervention, or exposure being studied.
C = Comparator: the comparison treatment, control, or alternative.
O = Outcome: the clinical outcome or result of interest.

Return exactly four fields:
P: ...
I: ...
C: ...
O: ...


Clinical question:
In adults with microsatellite instability-high (MSI-H) or mismatch repair-deficient (dMMR) metastatic colorectal cancer, does first-line pembrolizumab compared with standard chemotherapy (mFOLFOX6 or FOLFIRI) ± targeted therapy improve progression-free survival?<end_of_turn>
<start_of_turn>model

POSITIVE: P: Adults with MSI-H/dMMR metastatic colorectal cancer
I: Pembrolizumab monotherapy
C: Standard combination chemotherapy (mFOLFOX6/FOLFIRI ± bevacizumab/cetuxima

# **Part A : Rank 1 RePS**

## **Steering Vector rank 1**

In [34]:
class SteeringVector(nn.Module):
    """Rank-1 steering vector.
    'add'  mode (eq. 9):  h <- h + alpha * w1 + b1
    'null' mode (eq. 7):  h <- h - (ReLU(h . w1) / ||w1||^2) * w1
    """
    def __init__(self, dim, dtype=torch.float32):
        super().__init__()
        self.w1 = nn.Parameter(torch.randn(dim, dtype=dtype) * 0.01)
        self.b1 = nn.Parameter(torch.zeros(dim, dtype=dtype))

    def forward(self, h, alpha=0.0, mode="add"):
        w1 = self.w1.to(device=h.device, dtype=h.dtype)
        if mode == "add":
            b1 = self.b1.to(device=h.device, dtype=h.dtype)
            return h + alpha * w1 + b1
        elif mode == "null":
            # project out the (ReLU-gated) component along w1, per eq. 7
            dot = torch.einsum("...d,d->...", h, w1)          # (..., )
            gated = F.relu(dot) / (w1.norm() ** 2 + 1e-8)      # (..., )
            return h - gated.unsqueeze(-1) * w1
        else:
            raise ValueError(mode)


class SteeringHook:
    """Forward hook that injects a SteeringVector into a decoder layer's output."""
    def __init__(self, steering_vector: SteeringVector):
        self.sv = steering_vector
        self.enabled = False
        self.alpha = 0.0
        self.mode = "add"     # "add" for Phi_Steer, "null" for Phi_Null
        self.handle = None

    def __call__(self, module, inputs, output):
        if not self.enabled:
            return output
        if isinstance(output, tuple):
            hidden_states = self.sv(output[0], self.alpha, self.mode)
            return (hidden_states,) + output[1:]
        else:
            return self.sv(output, self.alpha, self.mode)

    def attach(self, layer_module):
        self.handle = layer_module.register_forward_hook(self)
        return self.handle

    def remove(self):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None

In [35]:
STEER_LAYER = max(1, int(NUM_LAYERS * 0.4))
layer_device = next(model.model.layers[STEER_LAYER].parameters()).device
print(f"Hooking layer index {STEER_LAYER} / {NUM_LAYERS} (on {layer_device})")

steering_vector = SteeringVector(HIDDEN_DIM, dtype=torch.float32).to(layer_device)
steering_hook = SteeringHook(steering_vector)
steering_hook.attach(model.model.layers[STEER_LAYER])

n_trainable = sum(p.numel() for p in steering_vector.parameters() if p.requires_grad)
print(f"Trainable steering-vector params: {n_trainable}  (2 x {HIDDEN_DIM})")

Hooking layer index 7 / 18 (on cuda:1)
Trainable steering-vector params: 4096  (2 x 2048)


## **Objective Function**

In [36]:
def seq_logprob(prompt_ids, response_ids, alpha, device, mode="add", use_hook=True):
    """Length-normalized log p(response | prompt).
    use_hook=False -> reference (unsteered) model likelihood.
    """
    input_ids = torch.cat([prompt_ids, response_ids], dim=0).unsqueeze(0).to(device)
    attention_mask = torch.ones_like(input_ids)

    steering_hook.enabled = use_hook
    steering_hook.alpha = alpha
    steering_hook.mode = mode
    out = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = out.logits

    shift_logits = logits[:, :-1, :].float()
    shift_labels = input_ids[:, 1:]
    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_logp = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)

    P = prompt_ids.shape[0]
    R = response_ids.shape[0]
    resp_logp = token_logp[:, P - 1 : P - 1 + R]
    steering_hook.enabled = False   # always leave disabled after use
    return resp_logp.sum() / R


def build_pair(example, device):
    prompt_ids = tokenizer(example["prompt"], return_tensors="pt", add_special_tokens=True).input_ids[0]
    pos_ids = tokenizer(example["positive"], return_tensors="pt", add_special_tokens=False).input_ids[0]
    neg_ids = tokenizer(example["negative"], return_tensors="pt", add_special_tokens=False).input_ids[0]
    return prompt_ids, pos_ids, neg_ids

## **Training**

In [37]:
TRAIN_FACTORS = [0.1, 0.15, 0.2, 0.25, 0.3, 0.5, 0.75, 1.0, 2.0, 4.0]
LR = 5e-4
EPOCHS = 8
GRAD_CLIP = 1.0
WEIGHT_DECAY = 0.01
PATIENCE = 2

optimizer = torch.optim.AdamW(steering_vector.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

@torch.no_grad()
def precompute_reference_logprobs(examples, device):
    """log p(y|x) and log p(y^c|x) under the frozen, unsteered reference model.
    Cached once since the base model never changes (reference-free RePS only
    uses these to derive the beta weighting terms, not as a KL anchor)."""
    cache = {}
    for i, ex in enumerate(examples):
        prompt_ids, pos_ids, neg_ids = build_pair(ex, device)
        logp_pos_ref = seq_logprob(prompt_ids, pos_ids, alpha=0.0, device=device, use_hook=False)
        logp_neg_ref = seq_logprob(prompt_ids, neg_ids, alpha=0.0, device=device, use_hook=False)
        cache[i] = (logp_pos_ref.item(), logp_neg_ref.item())
    return cache

def reps_loss(example, ref_logps, alpha, device):
    prompt_ids, pos_ids, neg_ids = build_pair(example, device)
    logp_pos_ref, logp_neg_ref = ref_logps

    # beta weights from the reference model (eq. 5 / eq. 6)
    beta_pos = max(logp_neg_ref - logp_pos_ref, 1.0)   # weights Delta+
    beta_neg = max(logp_pos_ref - logp_neg_ref, 1.0)   # weights Delta-

    # Delta+ : winning=positive(steered), losing=negative(original), under Phi_Steer (eq. 5)
    logp_pos_steer = seq_logprob(prompt_ids, pos_ids, alpha, device, mode="add", use_hook=True)
    logp_neg_steer = seq_logprob(prompt_ids, neg_ids, alpha, device, mode="add", use_hook=True)
    delta_pos = beta_pos * logp_pos_steer - logp_neg_steer

    # Delta- : winning=negative(original), losing=positive(steered), under Phi_Null (eq. 6)
    logp_neg_null = seq_logprob(prompt_ids, neg_ids, alpha=0.0, device=device, mode="null", use_hook=True)
    logp_pos_null = seq_logprob(prompt_ids, pos_ids, alpha=0.0, device=device, mode="null", use_hook=True)
    delta_neg = beta_neg * logp_neg_null - logp_pos_null

    loss = -F.logsigmoid(delta_pos) - F.logsigmoid(delta_neg)
    return loss, delta_pos.item(), delta_neg.item()

@torch.no_grad()
def evaluate(examples, ref_cache, alpha, device):
    losses, margins = [], []
    for i, ex in enumerate(examples):
        loss, d_pos, d_neg = reps_loss(ex, ref_cache[i], alpha, device)
        losses.append(loss.item())
        margins.append(d_pos)  # track steering margin as before
    acc = float(np.mean([m > 0 for m in margins]))
    return float(np.mean(losses)), float(np.mean(margins)), acc

In [38]:
print("Precomputing reference (unsteered) log-probs...")
train_ref_cache = precompute_reference_logprobs(train_examples, layer_device)
val_ref_cache = precompute_reference_logprobs(val_examples, layer_device)

history = {"train_loss": [], "val_loss": [], "val_margin": [], "val_acc": [], "vec_norm": [], "bias_norm": []}
best_val_loss = float("inf")
best_state = None
epochs_since_improve = 0

for epoch in range(EPOCHS):
    idx = list(range(len(train_examples)))
    random.shuffle(idx)
    epoch_losses = []
    for i in idx:
        ex = train_examples[i]
        alpha = random.choice(TRAIN_FACTORS)

        optimizer.zero_grad()
        loss, _, _ = reps_loss(ex, train_ref_cache[i], alpha, layer_device)

        if not torch.isfinite(loss):
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(steering_vector.parameters(), GRAD_CLIP)
        optimizer.step()
        epoch_losses.append(loss.item())

    train_loss = float(np.mean(epoch_losses)) if epoch_losses else float("nan")
    val_loss, val_margin, val_acc = evaluate(val_examples, val_ref_cache,
                                              alpha=TRAIN_FACTORS[len(TRAIN_FACTORS)//2],
                                              device=layer_device)
    vec_norm = steering_vector.w1.norm().item()
    bias_norm = steering_vector.b1.norm().item()
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_margin"].append(val_margin)
    history["val_acc"].append(val_acc)
    history["vec_norm"].append(vec_norm)
    history["bias_norm"].append(bias_norm)
    print(f"epoch {epoch+1:02d}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
          f"val_margin={val_margin:+.4f}  val_pref_acc={val_acc:.2f}  ||w1||={vec_norm:.3f} ||b1||={bias_norm:.3f}")

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_state = copy.deepcopy(steering_vector.state_dict())
        epochs_since_improve = 0
    else:
        epochs_since_improve += 1
        if epochs_since_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}.")
            break

if best_state is not None:
    steering_vector.load_state_dict(best_state)
    print(f"Restored best checkpoint (val_loss={best_val_loss:.4f}, ||w1||={steering_vector.w1.norm().item():.3f})")

Precomputing reference (unsteered) log-probs...
epoch 01/8  train_loss=1.5701  val_loss=0.6037  val_margin=+0.4134  val_pref_acc=0.60  ||w1||=0.589 ||b1||=0.610
epoch 02/8  train_loss=0.9790  val_loss=0.3458  val_margin=+1.0859  val_pref_acc=0.90  ||w1||=0.714 ||b1||=1.468
epoch 03/8  train_loss=0.6716  val_loss=0.1741  val_margin=+1.9055  val_pref_acc=1.00  ||w1||=0.838 ||b1||=2.124
epoch 04/8  train_loss=0.5140  val_loss=0.0623  val_margin=+3.0287  val_pref_acc=1.00  ||w1||=0.923 ||b1||=2.655
epoch 05/8  train_loss=0.4071  val_loss=0.0306  val_margin=+3.8238  val_pref_acc=1.00  ||w1||=0.968 ||b1||=2.925
epoch 06/8  train_loss=0.3638  val_loss=0.0170  val_margin=+4.4963  val_pref_acc=1.00  ||w1||=1.000 ||b1||=3.141
epoch 07/8  train_loss=0.3052  val_loss=0.0112  val_margin=+4.9532  val_pref_acc=1.00  ||w1||=1.016 ||b1||=3.287
epoch 08/8  train_loss=0.2768  val_loss=0.0076  val_margin=+5.3973  val_pref_acc=1.00  ||w1||=1.039 ||b1||=3.417
Restored best checkpoint (val_loss=0.0076, ||w1|

## **Best Steering Vector**

In [39]:
import re

@torch.no_grad()
def generate(prompt, alpha=None, max_new_tokens=100):
    """Greedy-decode a continuation, optionally with the steering vector enabled."""
    steering_hook.enabled = alpha is not None
    steering_hook.alpha = alpha if alpha is not None else 0.0
    steering_hook.mode = "add"

    inputs = tokenizer(prompt, return_tensors="pt").to(layer_device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    steering_hook.enabled = False
    return text


def extract_pico_fields(text):
    """Pull P:/I:/C:/O: lines out of a (possibly messy) generation."""
    fields = {}
    for key in ["P", "I", "C", "O"]:
        m = re.search(rf"^\s*{key}\s*:\s*(.+)$", text, flags=re.MULTILINE)
        fields[key] = m.group(1).strip() if m else ""
    return fields


def word_f1(a, b):
    """Token-overlap F1 between two field strings -- a cheap stand-in for the
    paper's LM-judge concept score, since we don't have gpt-4o-mini/Qwen budget
    to judge every candidate factor x example."""
    a_words = set(re.findall(r"[a-zA-Z0-9]+", a.lower()))
    b_words = set(re.findall(r"[a-zA-Z0-9]+", b.lower()))
    if not a_words or not b_words:
        return 0.0
    inter = len(a_words & b_words)
    precision = inter / len(b_words)
    recall = inter / len(a_words)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def repetition_penalty(text, n=3):
    """Fraction of repeated n-grams. RePS vectors pushed too hard degenerate into
    repeated fragments rather than gibberish tokens (paper Appendix H.1) -- this
    catches that failure mode as a crude fluency proxy."""
    tokens = text.split()
    if len(tokens) <= n:
        return 0.0
    ngrams = [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]
    return 1.0 - (len(set(ngrams)) / len(ngrams))


def pico_quality_score(generated, ground_truth_positive):
    """Harmonic mean of (format adherence, field-content overlap, non-degeneracy) --
    same spirit as AxBench's harmonic mean of (concept, instruct, fluency) scores,
    adapted to a task where we can check structured-field correctness directly
    instead of needing an LM judge."""
    gt_fields = extract_pico_fields(ground_truth_positive)
    gen_fields = extract_pico_fields(generated)

    present = sum(1 for k in gt_fields if gt_fields[k])
    if present == 0:
        return 0.0
    format_score = sum(1 for k in gt_fields if gt_fields[k] and gen_fields[k]) / present

    sims = [word_f1(gt_fields[k], gen_fields[k]) for k in gt_fields if gt_fields[k]]
    content_score = float(np.mean(sims)) if sims else 0.0

    fluency_score = 1.0 - repetition_penalty(generated)

    eps = 1e-4
    scores = [format_score + eps, content_score + eps, fluency_score + eps]
    harmonic = len(scores) / sum(1.0 / s for s in scores)
    return harmonic


In [42]:
# Split the held-out set the way AxBench splits its per-concept eval instructions:
# one half picks the factor, the other half is reserved purely for reporting, so the
# examples we act on and the examples we report results on never overlap.
random.shuffle(val_examples)

N_SELECT = 6

sel_examples = val_examples[:N_SELECT]
hold_examples = val_examples[N_SELECT:]

print(
    f"Selecting factor on {len(sel_examples)} example(s), "
    f"holding out {len(hold_examples)} example(s) for reporting."
)
'''
random.shuffle(val_examples)
mid = max(1, len(val_examples) // 2)
sel_examples = val_examples[:mid]
hold_examples = val_examples[mid:] if len(val_examples) > mid else val_examples[:mid]

print(f"Selecting factor on {len(sel_examples)} example(s), "
      f"holding out {len(hold_examples)} example(s) for reporting.")
'''
# Sweep stays inside (or just past the top of) TRAIN_FACTORS -- we don't test factors
# the vector never saw gradients for, and we do add a couple of values above the
# trained max to see if it extrapolates, matching how the paper's inference-time
# factor set extends beyond the training set on the high end only (Table 11).
candidate_factors = TRAIN_FACTORS

# Baseline (no steering) score, so we can confirm steering is actually helping
# rather than just picking the "least bad" factor.
baseline_scores = [pico_quality_score(generate(ex["prompt"], alpha=None), ex["positive"])
                    for ex in sel_examples]
baseline_quality = float(np.mean(baseline_scores))
print(f"unsteered baseline pico_quality={baseline_quality:.4f}\n")

factor_scores = {}
for f in candidate_factors:
    scores = [pico_quality_score(generate(ex["prompt"], alpha=f), ex["positive"])
              for ex in sel_examples]
    factor_scores[f] = float(np.mean(scores))
    print(f"alpha={f:>5.1f}  pico_quality={factor_scores[f]:.4f}")

BEST_ALPHA = max(factor_scores, key=factor_scores.get)
print(f"\nBest steering factor (by held-out generation quality): {BEST_ALPHA} "
      f"(pico_quality={factor_scores[BEST_ALPHA]:.4f} vs. baseline={baseline_quality:.4f})")

if factor_scores[BEST_ALPHA] <= baseline_quality:
    print("WARNING: best steered score does not beat the unsteered baseline on the "
          "selection split -- inspect generations below before trusting BEST_ALPHA.")


Selecting factor on 6 example(s), holding out 4 example(s) for reporting.
unsteered baseline pico_quality=0.6595

alpha=  0.1  pico_quality=0.6769
alpha=  0.1  pico_quality=0.6769
alpha=  0.2  pico_quality=0.6567
alpha=  0.2  pico_quality=0.6567
alpha=  0.3  pico_quality=0.6544
alpha=  0.5  pico_quality=0.6501
alpha=  0.8  pico_quality=0.6442
alpha=  1.0  pico_quality=0.6366
alpha=  2.0  pico_quality=0.5392
alpha=  4.0  pico_quality=0.3288
alpha=  6.0  pico_quality=0.2489

Best steering factor (by held-out generation quality): 0.1 (pico_quality=0.6769 vs. baseline=0.6595)


## **Generation**

In [43]:
for ex in hold_examples:
    print("=" * 80)
    print("PROMPT:", ex["prompt"].strip())
    print("-" * 80)

    unsteered = generate(ex["prompt"], alpha=None)
    steered = generate(ex["prompt"], alpha=BEST_ALPHA)

    print("UNSTEERED:")
    print(unsteered)
    print(f"  pico_quality={pico_quality_score(unsteered, ex['positive']):.4f}")
    print("-" * 80)
    print(f"STEERED (learned vector, alpha={BEST_ALPHA}):")
    print(steered)
    print(f"  pico_quality={pico_quality_score(steered, ex['positive']):.4f}")
    print("-" * 80)
    print("POSITIVE (ground truth):")
    print(ex["positive"].replace(tokenizer.eos_token, ""))


PROMPT: <bos><start_of_turn>user
You are given a clinical question.

Extract the question into the PICO framework:

P = Population/Patient: the patient population or condition being studied.
I = Intervention/Exposure: the treatment, intervention, or exposure being studied.
C = Comparator: the comparison treatment, control, or alternative.
O = Outcome: the clinical outcome or result of interest.

Return exactly four fields:
P: ...
I: ...
C: ...
O: ...


Clinical question:
In adults with active IgA Nephropathy, does sibeprenlimab compared with placebo reduce 24-hour proteinuria at 9 months?<end_of_turn>
<start_of_turn>model
--------------------------------------------------------------------------------
UNSTEERED:
P: Patient population or condition being studied.
I: Sibeprenlimab.
C: Placebo.
O: 24-hour proteinuria.
  pico_quality=0.6819
--------------------------------------------------------------------------------
STEERED (learned vector, alpha=0.1):
P: Adult patients with IgA Nephrop

In [45]:
for ex in hold_examples:
    print("=" * 80)
    print("PROMPT:", ex["prompt"].strip())
    print("-" * 80)

    unsteered = generate(ex["prompt"], alpha=None)
    steered = generate(ex["prompt"], alpha=0.5)

    print("UNSTEERED:")
    print(unsteered)
    print(f"  pico_quality={pico_quality_score(unsteered, ex['positive']):.4f}")
    print("-" * 80)
    print(f"STEERED (learned vector, alpha={0.5}):")
    print(steered)
    print(f"  pico_quality={pico_quality_score(steered, ex['positive']):.4f}")
    print("-" * 80)
    print("POSITIVE (ground truth):")
    print(ex["positive"].replace(tokenizer.eos_token, ""))


PROMPT: <bos><start_of_turn>user
You are given a clinical question.

Extract the question into the PICO framework:

P = Population/Patient: the patient population or condition being studied.
I = Intervention/Exposure: the treatment, intervention, or exposure being studied.
C = Comparator: the comparison treatment, control, or alternative.
O = Outcome: the clinical outcome or result of interest.

Return exactly four fields:
P: ...
I: ...
C: ...
O: ...


Clinical question:
In adults with active IgA Nephropathy, does sibeprenlimab compared with placebo reduce 24-hour proteinuria at 9 months?<end_of_turn>
<start_of_turn>model
--------------------------------------------------------------------------------
UNSTEERED:
P: Patient population or condition being studied.
I: Sibeprenlimab.
C: Placebo.
O: 24-hour proteinuria.
  pico_quality=0.6819
--------------------------------------------------------------------------------
STEERED (learned vector, alpha=0.5):
P: Adult patients with IgA Nephrop

In [ ]:
torch.save(
    {
        "w1": steering_vector.w1.detach().cpu(),
        "b1": steering_vector.b1.detach().cpu(),
        "layer": STEER_LAYER,
        "best_alpha": BEST_ALPHA,
        "model_name": MODEL_NAME,
        "objective": "preference (positive vs negative)",
    },
    "/kaggle/working/pico_steering_vector_rank1.pt",
)
print("Saved /kaggle/working/pico_steering_vector_rank1.pt")

# **Part B: RePS Rank 4 LoRA**

In [52]:
class Rank4LoRAModule(nn.Module):
    def __init__(self, dim, rank=4, dtype=torch.float32):
        super().__init__()
        self.rank = rank
        self.w1 = nn.Parameter(torch.randn(dim, rank, dtype=dtype) * 0.01)
        self.w2 = nn.Parameter(torch.zeros(dim, rank, dtype=dtype))

    def forward(self, h, alpha=1.0):
        w1 = self.w1.to(device=h.device, dtype=h.dtype)
        w2 = self.w2.to(device=h.device, dtype=h.dtype)
        # Scale by rank (1 / self.rank) to prevent activation explosion
        delta = torch.matmul(torch.matmul(h, w1), w2.T) / self.rank
        return h + alpha * delta


class MultiLayerLoRAHook:
    """Hooks Rank-4 LoRA adapters across multiple decoder layers (e.g. layers 5, 10, 15, 20)."""
    def __init__(self, lora_modules: nn.ModuleDict):
        self.lora_modules = lora_modules
        self.enabled = False
        self.alpha = 1.0
        self.handles = []

    def make_hook(self, layer_idx_str):
        def hook(module, inputs, output):
            if not self.enabled:
                return output
            mod = self.lora_modules[layer_idx_str]
            if isinstance(output, tuple):
                hidden_states = mod(output[0], alpha=self.alpha)
                return (hidden_states,) + output[1:]
            else:
                return mod(output, alpha=self.alpha)
        return hook

    def attach(self, model):
        for layer_str, mod in self.lora_modules.items():
            layer_idx = int(layer_str)
            handle = model.model.layers[layer_idx].register_forward_hook(self.make_hook(layer_str))
            self.handles.append(handle)

    def remove(self):
        for h in self.handles:
            h.remove()
        self.handles = []

In [53]:
# Define 4 target layers as specified in Table 5 for Gemma-2-2B
TARGET_LAYERS = [5, 10, 15]
lora_modules_dict = nn.ModuleDict()

for l_idx in TARGET_LAYERS:
    layer_device = next(model.model.layers[l_idx].parameters()).device
    lora_modules_dict[str(l_idx)] = Rank4LoRAModule(HIDDEN_DIM, rank=4, dtype=torch.float32).to(layer_device)

lora_hook = MultiLayerLoRAHook(lora_modules_dict)
lora_hook.attach(model)

n_trainable = sum(p.numel() for p in lora_modules_dict.parameters() if p.requires_grad)
print(f"Hooked layers {TARGET_LAYERS}. Total trainable Rank-4 LoRA params: {n_trainable}")

Hooked layers [5, 10, 15]. Total trainable Rank-4 LoRA params: 49152


## **Objective Function**

In [54]:
def seq_logprob(prompt_ids, response_ids, alpha, device, use_hook=True):
    """Length-normalized log p(response | prompt).
    use_hook=False -> reference (unsteered) model likelihood.
    """
    input_ids = torch.cat([prompt_ids, response_ids], dim=0).unsqueeze(0).to(device)
    attention_mask = torch.ones_like(input_ids)

    lora_hook.enabled = use_hook
    lora_hook.alpha = alpha
    out = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = out.logits

    shift_logits = logits[:, :-1, :].float()
    shift_labels = input_ids[:, 1:]
    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_logp = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)

    P = prompt_ids.shape[0]
    R = response_ids.shape[0]
    resp_logp = token_logp[:, P - 1 : P - 1 + R]
    lora_hook.enabled = False   # always leave disabled after use
    return resp_logp.sum() / R


def build_pair(example, device):
    prompt_ids = tokenizer(example["prompt"], return_tensors="pt", add_special_tokens=True).input_ids[0]
    pos_ids = tokenizer(example["positive"], return_tensors="pt", add_special_tokens=False).input_ids[0]
    neg_ids = tokenizer(example["negative"], return_tensors="pt", add_special_tokens=False).input_ids[0]
    return prompt_ids, pos_ids, neg_ids

## **Training**

In [55]:
TRAIN_FACTORS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.3, 0.4, 0.5, 0.75, 1.0, 1.2]
LR = 1e-4
EPOCHS = 8
GRAD_CLIP = 1.0
WEIGHT_DECAY = 0.01
PATIENCE = 2

optimizer = torch.optim.AdamW(lora_modules_dict.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

@torch.no_grad()
def precompute_reference_logprobs(examples, device):
    """log p(y|x) and log p(y^c|x) under frozen reference model."""
    cache = {}
    for i, ex in enumerate(examples):
        prompt_ids, pos_ids, neg_ids = build_pair(ex, device)
        logp_pos_ref = seq_logprob(prompt_ids, pos_ids, alpha=0.0, device=device, use_hook=False)
        logp_neg_ref = seq_logprob(prompt_ids, neg_ids, alpha=0.0, device=device, use_hook=False)
        cache[i] = (logp_pos_ref.item(), logp_neg_ref.item())
    return cache

def reps_loss(example, ref_logps, alpha, device):
    prompt_ids, pos_ids, neg_ids = build_pair(example, device)
    logp_pos_ref, logp_neg_ref = ref_logps

    # beta weights from reference model (Eq. 5 / Eq. 6)
    beta_pos = max(logp_neg_ref - logp_pos_ref, 1.0)
    beta_neg = max(logp_pos_ref - logp_neg_ref, 1.0)

    # Delta+ : positive factor (+alpha) under Phi_LoRA
    logp_pos_steer = seq_logprob(prompt_ids, pos_ids, alpha=+alpha, device=device, use_hook=True)
    logp_neg_steer = seq_logprob(prompt_ids, neg_ids, alpha=+alpha, device=device, use_hook=True)
    delta_pos = beta_pos * logp_pos_steer - logp_neg_steer

    # Delta- : negative factor (-alpha) for LoRA RePS objective (Sec. 3.3)
    logp_neg_null = seq_logprob(prompt_ids, neg_ids, alpha=-alpha, device=device, use_hook=True)
    logp_pos_null = seq_logprob(prompt_ids, pos_ids, alpha=-alpha, device=device, use_hook=True)
    delta_neg = beta_neg * logp_neg_null - logp_pos_null

    loss = -F.logsigmoid(delta_pos) - F.logsigmoid(delta_neg)
    return loss, delta_pos.item(), delta_neg.item()

@torch.no_grad()
def evaluate(examples, ref_cache, alpha, device):
    losses, margins = [], []
    for i, ex in enumerate(examples):
        loss, d_pos, d_neg = reps_loss(ex, ref_cache[i], alpha, device)
        if torch.isfinite(loss) and math.isfinite(d_pos):
            losses.append(loss.item())
            margins.append(d_pos)
            
    if not losses:
        return float("nan"), float("nan"), 0.0
        
    acc = float(np.mean([m > 0 for m in margins]))
    return float(np.mean(losses)), float(np.mean(margins)), acc

In [56]:
main_device = next(model.model.layers[TARGET_LAYERS[0]].parameters()).device
print("Precomputing reference (unsteered) log-probs...")
train_ref_cache = precompute_reference_logprobs(train_examples, main_device)
val_ref_cache = precompute_reference_logprobs(val_examples, main_device)

history = {"train_loss": [], "val_loss": [], "val_margin": [], "val_acc": []}
best_val_loss = float("inf")
best_state = None
epochs_since_improve = 0

for epoch in range(EPOCHS):
    idx = list(range(len(train_examples)))
    random.shuffle(idx)
    epoch_losses = []
    for i in idx:
        ex = train_examples[i]
        alpha = random.choice(TRAIN_FACTORS)

        optimizer.zero_grad()
        loss, _, _ = reps_loss(ex, train_ref_cache[i], alpha, main_device)

        if not torch.isfinite(loss):
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(lora_modules_dict.parameters(), GRAD_CLIP)
        optimizer.step()
        epoch_losses.append(loss.item())

    train_loss = float(np.mean(epoch_losses)) if epoch_losses else float("nan")
    val_loss, val_margin, val_acc = evaluate(val_examples, val_ref_cache,
                                              alpha=TRAIN_FACTORS[len(TRAIN_FACTORS)//2],
                                              device=main_device)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_margin"].append(val_margin)
    history["val_acc"].append(val_acc)
    print(f"epoch {epoch+1:02d}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
          f"val_margin={val_margin:+.4f}  val_pref_acc={val_acc:.2f}")

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_state = copy.deepcopy(lora_modules_dict.state_dict())
        epochs_since_improve = 0
    else:
        epochs_since_improve += 1
        if epochs_since_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}.")
            break

if best_state is not None:
    lora_modules_dict.load_state_dict(best_state)
    print(f"Restored best checkpoint (val_loss={best_val_loss:.4f})")

Precomputing reference (unsteered) log-probs...
epoch 01/8  train_loss=2.1246  val_loss=1.3764  val_margin=+0.0565  val_pref_acc=0.60
epoch 02/8  train_loss=1.8143  val_loss=1.0699  val_margin=+0.3902  val_pref_acc=0.60
epoch 03/8  train_loss=1.5343  val_loss=0.7586  val_margin=+0.8489  val_pref_acc=0.90
epoch 04/8  train_loss=1.0578  val_loss=0.5312  val_margin=+1.3484  val_pref_acc=0.90
epoch 05/8  train_loss=0.9651  val_loss=0.3481  val_margin=+1.9756  val_pref_acc=1.00
epoch 06/8  train_loss=0.6773  val_loss=0.2181  val_margin=+2.6715  val_pref_acc=1.00
epoch 07/8  train_loss=0.7808  val_loss=0.1332  val_margin=+3.4373  val_pref_acc=1.00
epoch 08/8  train_loss=0.8260  val_loss=0.0796  val_margin=+4.3522  val_pref_acc=1.00
Restored best checkpoint (val_loss=0.0796)


## **Best Steering Vector**

In [57]:
@torch.no_grad()
def generate(prompt, alpha=None, max_new_tokens=100):
    """Greedy-decode a continuation, optionally with LoRA adapters enabled."""
    lora_hook.enabled = alpha is not None
    lora_hook.alpha = alpha if alpha is not None else 0.0

    inputs = tokenizer(prompt, return_tensors="pt").to(main_device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    lora_hook.enabled = False
    return text


def extract_pico_fields(text):
    """Pull P:/I:/C:/O: lines out of a generation."""
    fields = {}
    for key in ["P", "I", "C", "O"]:
        m = re.search(rf"^\s*{key}\s*:\s*(.+)$", text, flags=re.MULTILINE)
        fields[key] = m.group(1).strip() if m else ""
    return fields


def word_f1(a, b):
    a_words = set(re.findall(r"[a-zA-Z0-9]+", a.lower()))
    b_words = set(re.findall(r"[a-zA-Z0-9]+", b.lower()))
    if not a_words or not b_words:
        return 0.0
    inter = len(a_words & b_words)
    precision = inter / len(b_words)
    recall = inter / len(a_words)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def repetition_penalty(text, n=3):
    tokens = text.split()
    if len(tokens) <= n:
        return 0.0
    ngrams = [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]
    return 1.0 - (len(set(ngrams)) / len(ngrams))


def pico_quality_score(generated, ground_truth_positive):
    gt_fields = extract_pico_fields(ground_truth_positive)
    gen_fields = extract_pico_fields(generated)

    present = sum(1 for k in gt_fields if gt_fields[k])
    if present == 0:
        return 0.0
    format_score = sum(1 for k in gt_fields if gt_fields[k] and gen_fields[k]) / present

    sims = [word_f1(gt_fields[k], gen_fields[k]) for k in gt_fields if gt_fields[k]]
    content_score = float(np.mean(sims)) if sims else 0.0

    fluency_score = 1.0 - repetition_penalty(generated)

    eps = 1e-4
    scores = [format_score + eps, content_score + eps, fluency_score + eps]
    harmonic = len(scores) / sum(1.0 / s for s in scores)
    return harmonic

In [58]:
random.shuffle(val_examples)
N_SELECT = 6
sel_examples = val_examples[:N_SELECT]
hold_examples = val_examples[N_SELECT:]
print(
    f"Selecting factor on {len(sel_examples)} example(s), "
    f"holding out {len(hold_examples)} example(s) for reporting."
)

'''
random.shuffle(val_examples)
mid = max(1, len(val_examples) // 2)
sel_examples = val_examples[:mid]
hold_examples = val_examples[mid:] if len(val_examples) > mid else val_examples[:mid]
print(f"Selecting factor on {len(sel_examples)} example(s), "
      f"holding out {len(hold_examples)} example(s) for reporting.")
'''

# Candidate inference factors specified for LoRA in Table 11
#candidate_factors = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.5, 3.0, 4.0, 5.0]
candidate_factors = [0.05, 0.10, 0.15, 0.20, 0.25, 0.3, 0.4, 0.5, 0.75, 1.0, 1.2]

baseline_scores = [pico_quality_score(generate(ex["prompt"], alpha=None), ex["positive"])
                    for ex in sel_examples]
baseline_quality = float(np.mean(baseline_scores))
print(f"unsteered baseline pico_quality={baseline_quality:.4f}\n")

factor_scores = {}
for f in candidate_factors:
    scores = [pico_quality_score(generate(ex["prompt"], alpha=f), ex["positive"])
              for ex in sel_examples]
    factor_scores[f] = float(np.mean(scores))
    print(f"alpha={f:>5.1f}  pico_quality={factor_scores[f]:.4f}")

BEST_ALPHA = max(factor_scores, key=factor_scores.get)
print(f"\nBest steering factor (by held-out generation quality): {BEST_ALPHA} "
      f"(pico_quality={factor_scores[BEST_ALPHA]:.4f} vs. baseline={baseline_quality:.4f})")

if factor_scores[BEST_ALPHA] <= baseline_quality:
    print("WARNING: best steered score does not beat the unsteered baseline on the "
          "selection split -- inspect generations below before trusting BEST_ALPHA.")

Selecting factor on 6 example(s), holding out 4 example(s) for reporting.
unsteered baseline pico_quality=0.6535

alpha=  0.1  pico_quality=0.6928
alpha=  0.1  pico_quality=0.6980
alpha=  0.1  pico_quality=0.6791
alpha=  0.2  pico_quality=0.6390
alpha=  0.2  pico_quality=0.6140
alpha=  0.3  pico_quality=0.5624
alpha=  0.4  pico_quality=0.5641
alpha=  0.5  pico_quality=0.4825
alpha=  0.8  pico_quality=0.0327
alpha=  1.0  pico_quality=0.0003
alpha=  1.2  pico_quality=0.0003

Best steering factor (by held-out generation quality): 0.1 (pico_quality=0.6980 vs. baseline=0.6535)


## **Generation**

In [59]:
for ex in hold_examples:
    print("=" * 80)
    print("PROMPT:", ex["prompt"].strip())
    print("-" * 80)

    unsteered = generate(ex["prompt"], alpha=None)
    steered = generate(ex["prompt"], alpha=BEST_ALPHA)

    print("UNSTEERED:")
    print(unsteered)
    print(f"  pico_quality={pico_quality_score(unsteered, ex['positive']):.4f}")
    print("-" * 80)
    print(f"STEERED (learned Rank-4 LoRA, alpha={BEST_ALPHA}):")
    print(steered)
    print(f"  pico_quality={pico_quality_score(steered, ex['positive']):.4f}")
    print("-" * 80)
    print("POSITIVE (ground truth):")
    print(ex["positive"].replace(tokenizer.eos_token, ""))

PROMPT: <bos><start_of_turn>user
You are given a clinical question.

Extract the question into the PICO framework:

P = Population/Patient: the patient population or condition being studied.
I = Intervention/Exposure: the treatment, intervention, or exposure being studied.
C = Comparator: the comparison treatment, control, or alternative.
O = Outcome: the clinical outcome or result of interest.

Return exactly four fields:
P: ...
I: ...
C: ...
O: ...


Clinical question:
In patients with confirmed Staphylococcus aureus bacteremia, does early switching to oral antibiotic therapy compared with continuing an all-intravenous course affect clinical success at 90 days?<end_of_turn>
<start_of_turn>model
--------------------------------------------------------------------------------
UNSTEERED:
P: Patient population or condition being studied.
I: Early switching to oral antibiotic therapy compared with continuing an all-intravenous course.
C: All-intravenous course.
O: Clinical success at 90 d

In [60]:
for ex in hold_examples:
    print("=" * 80)
    print("PROMPT:", ex["prompt"].strip())
    print("-" * 80)

    unsteered = generate(ex["prompt"], alpha=None)
    steered = generate(ex["prompt"], alpha=0.2)

    print("UNSTEERED:")
    print(unsteered)
    print(f"  pico_quality={pico_quality_score(unsteered, ex['positive']):.4f}")
    print("-" * 80)
    print(f"STEERED (learned Rank-4 LoRA, alpha={0.2}):")
    print(steered)
    print(f"  pico_quality={pico_quality_score(steered, ex['positive']):.4f}")
    print("-" * 80)
    print("POSITIVE (ground truth):")
    print(ex["positive"].replace(tokenizer.eos_token, ""))

PROMPT: <bos><start_of_turn>user
You are given a clinical question.

Extract the question into the PICO framework:

P = Population/Patient: the patient population or condition being studied.
I = Intervention/Exposure: the treatment, intervention, or exposure being studied.
C = Comparator: the comparison treatment, control, or alternative.
O = Outcome: the clinical outcome or result of interest.

Return exactly four fields:
P: ...
I: ...
C: ...
O: ...


Clinical question:
In patients with confirmed Staphylococcus aureus bacteremia, does early switching to oral antibiotic therapy compared with continuing an all-intravenous course affect clinical success at 90 days?<end_of_turn>
<start_of_turn>model
--------------------------------------------------------------------------------
UNSTEERED:
P: Patient population or condition being studied.
I: Early switching to oral antibiotic therapy compared with continuing an all-intravenous course.
C: All-intravenous course.
O: Clinical success at 90 d

In [ ]:
torch.save(
    {
        "lora_state_dict": lora_modules_dict.state_dict(),
        "target_layers": TARGET_LAYERS,
        "rank": 4,
        "best_alpha": BEST_ALPHA,
        "model_name": MODEL_NAME,
        "objective": "LoRA RePS (positive vs negative sampling)",
    },
    "/kaggle/working/pico_lora_r4.pt",
)
print("Saved /kaggle/working/pico_lora_r4.pt")